
# colab_16 — scGPT continued pretraining (CPT), aggregated regime · N=1 pilot

The scGPT arm of the CPT comparison. This notebook is the direct counterpart of `colab_11`
(Geneformer, aggregated regime): it continues pretraining the whole-human scGPT checkpoint on
the same glia substrate, under the same frozen donor split, with the same LoRA rank and the same
step budget — then re-embeds the substrate and runs **detector #1** (is the embedding changing?).
Evals #1/#2 are a separate notebook, exactly as `colab_11` handed off to `colab_12`.

**What is held fixed against the Geneformer run** — substrate (142,588 glia), donor split
(seed 32, 101/22/22 donors), LoRA rank 8 / alpha 16 / dropout 0.05, masking probability 0.15,
learning rate 5e-4, effective batch 32, 2,000 optimizer steps (ca. 0.674 epochs over the same
94,963 training cells). The intended difference is the model and its native input encoding.

**What is necessarily different, and why it matters.**

1. **scGPT's objective is masked *value* prediction, not masked *token* prediction.** Genes enter
   as symbols; their expression enters as a separate binned value. CPT masks the value and
   regresses it back (masked MSE), rather than reconstructing a rank-ordered token. This is the
   input-representation difference that makes the two FMs a genuine contrast rather than a
   duplicate run.
2. **The LoRA adapters sit entirely upstream of the readout.** The cell embedding is the
   `<cls>` position at the top of the encoder stack, and the expression decoder that produces the
   training loss is strictly downstream of it. Every adapted parameter in this run is therefore
   *visible* to detector #1 — there is no structural equivalent of the Geneformer arrangement
   where part of the adapted capacity sat past the extraction point. A drift/loss decoupling here
   could not be explained the same way.
3. **The embedding is stochastic.** scGPT truncates each cell to 1,200 gene tokens by *random
   sampling* when it has more, so embedding the same cell twice with the same weights does not
   give the same vector. Detector #1's noise floor is therefore not zero and cannot be inherited
   — §6a measures it in this run, twice, and §7a gates against the measured value.
4. **The magnitude anchor has to be recomputed in this space.** The within-donor substate
   reference (0.0365 micro / 0.0442 astro) is a distance in Geneformer's embedding space and says
   nothing about scGPT's. §7b recomputes it from the stored scGPT zero-shot embedding so the
   drift is reported as a percentage of a meaningful change *in the space it was measured in*.

**Inputs** — `micro_subset.h5ad` / `astro_subset.h5ad` (Drive), `outputs/donor_split.json`,
`scgpt_whole_human/` checkpoint (Drive), and
`scgpt/glia_scgpt_zeroshot.h5ad` (the zero-shot baseline this run drifts away from).

**Outputs** — a LoRA adapter directory, a post-CPT embedding `.h5ad`, and a
`scgpt_cpt_aggregated` entry in `outputs/audit_report.json`.


## 1 — Setup



### 1a — Run-control flags, Drive, repo, scGPT install, checkpoint

`SMOKE` is the single run-control flag. Every output path in the notebook is derived from it, so a
smoke run can waste time but cannot overwrite a real artifact. The subsample it applies is taken
in §3c — after the substrate and split checks have run against the full object, and before the
first expensive stage (tokenizing and embedding), which is where a plumbing rehearsal has to sit
to be worth anything.

scGPT is installed from a pinned commit with `--no-deps`: its declared dependency set is far wider
than what the load/embed/train path actually imports, and the extras fight Colab's native stack.
`peft` is pinned exactly, because this notebook depends on its LoRA merge semantics for
`nn.MultiheadAttention` and a version skew there would change results without raising anything.

**Restart the runtime after this cell**, every time -- every prior Colab notebook in this project has needed it after upgrading packages over Colab's base image, and there is no reason this one would be the exception.


In [ ]:
import os, subprocess, sys
from google.colab import drive

# ---------------------------------------------------------------- run-control
SMOKE = False           # True = plumbing rehearsal on a small subsample; all writes get a _SMOKE suffix
SMOKE_CAP = 40          # cells per (split x lineage x substate) group when SMOKE
SUFFIX = "_SMOKE" if SMOKE else ""
RUN_TAG = "seed0"

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"
if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
print("Python:", sys.version.split()[0])
print("repo commit:", subprocess.run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"],
                                     capture_output=True, text=True).stdout.strip())

# scGPT source only (--no-deps) at the CUDA-12.8-compatible commit; real runtime deps come from
# the requirements file. flash-attn is deliberately absent -> scGPT falls back to PyTorch
# attention, which is what the zero-shot baseline was computed under (colab_10).
SCGPT_COMMIT = "cebd6fae655b9c585a4807daa3ac31bb764f06b4"
!pip install --no-deps "git+https://github.com/bowang-lab/scGPT.git@{SCGPT_COMMIT}"
!pip install -r {REPO_PATH}/requirements_scgpt.txt

# Colab's base image ships torchao 0.10.0, but peft 0.19.1's `is_torchao_available()` RAISES on any
# version below 0.16.0 instead of returning False, so `get_peft_model()` dies at §5a. Nothing in the
# scGPT stack imports torchao, so removing it makes that capability probe return False cleanly. The
# Geneformer notebooks carry the same fix; it was not inherited here because this install cell was
# written fresh for scGPT.
!pip uninstall -y torchao

# peft's merge semantics are correctness-critical here (see §5a) -- assert, don't assume.
import peft
PEFT_PIN = "0.19.1"
assert peft.__version__ == PEFT_PIN, (
    f"peft {peft.__version__} != pinned {PEFT_PIN}; this notebook depends on its "
    "nn.MultiheadAttention LoRA merge behaviour -- do not proceed on a different version.")
print("peft:", peft.__version__)

# --- whole-human pretrained checkpoint -> Drive (cached across runs) ---------------------
MODEL_DIR  = os.path.join(DRIVE_ROOT, "scgpt_whole_human")
CKPT_FILES = ["vocab.json", "args.json", "best_model.pt"]

def _have_ckpt(d):
    return all(os.path.exists(os.path.join(d, f)) for f in CKPT_FILES)

if not _have_ckpt(MODEL_DIR):
    os.makedirs(MODEL_DIR, exist_ok=True)
    WHOLE_HUMAN_FOLDER = "1oWh_-ZRdhtoGQ2Fw24HP41FgLoomVo-y"   # scGPT README pretrained table
    !pip install -q gdown
    import gdown
    gdown.download_folder(id=WHOLE_HUMAN_FOLDER, output=MODEL_DIR, quiet=False, use_cookies=False)
    hits = [dp for dp, _, fs in os.walk(MODEL_DIR) if "best_model.pt" in fs]
    assert hits, f"best_model.pt not found under {MODEL_DIR} after download (Drive quota? download manually)"
    MODEL_DIR = hits[0]
assert _have_ckpt(MODEL_DIR), f"checkpoint incomplete in {MODEL_DIR}: need {CKPT_FILES}"
print("scGPT commit:", SCGPT_COMMIT[:7], "| checkpoint dir:", MODEL_DIR)

# CPT on CPU is not viable at this scale.
import torch
assert torch.cuda.is_available(), "no CUDA GPU -- select a GPU runtime before running CPT"
print("GPU:", torch.cuda.get_device_name(0))
print(f"\nSMOKE={SMOKE} | output suffix {SUFFIX!r}")


## 2 — Environment capture



### 2a — pip freeze + env JSON

The exact resolved stack for this run, written next to the other notebooks' snapshots. `_ver`
imports each package rather than reading installed metadata, so a broken import chain surfaces
here as a printed failure rather than a silent `None`.


In [ ]:
import json, platform, subprocess, sys
from datetime import date

NOTEBOOK_ID = "colab_16"
TODAY = date.today().isoformat()
VERSIONS_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(VERSIONS_DIR, exist_ok=True)

FREEZE_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze{SUFFIX}.txt")
!pip freeze > {FREEZE_PATH}

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

def _ver(mod):
    try:
        m = __import__(mod)
    except Exception as e:
        print(f"  [_ver] import '{mod}' FAILED: {type(e).__name__}: {e}")
        return None
    try:
        return m.__version__
    except AttributeError:
        import importlib.metadata as ilm
        try:
            return ilm.version(mod)
        except Exception:
            return None

env_snapshot = {
    "notebook_id":    NOTEBOOK_ID,
    "date":           TODAY,
    "smoke":          bool(SMOKE),
    "python_version": sys.version,
    "platform":       platform.platform(),
    "os_release":     platform.release(),
    "gpu":            _run(["nvidia-smi", "-L"]),
    "cuda":           _run(["nvcc", "--version"]),
    "git_commit":     _run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"]),
    "scgpt_commit":     SCGPT_COMMIT,
    "scgpt_version":    _ver("scgpt"),
    "peft_version":     _ver("peft"),
    "scanpy_version":   _ver("scanpy"),
    "anndata_version":  _ver("anndata"),
    "torch_version":    _ver("torch"),
    "numpy_version":    _ver("numpy"),
    "datasets_version": _ver("datasets"),
    "model_checkpoint": os.path.basename(MODEL_DIR),
}
ENV_JSON_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_env{SUFFIX}.json")
with open(ENV_JSON_PATH, "w") as f:
    json.dump(env_snapshot, f, indent=2)
print(json.dumps(env_snapshot, indent=2))


## 3 — Substrate, schema, frozen split



### 3a — Load both labelled subsets, concatenate, guard raw counts, apply scGPT's input transform

The substrate is rebuilt from the two labelled subsets rather than loaded from a saved file, so
`cell_index` is regenerated deterministically and matches every other notebook that rebuilds it
the same way. `cell_index` is the only key used to realign embeddings later — row order is never
trusted on its own.

The raw-counts guard matters more than it looks: scGPT's input pipeline expects to start from raw
counts and applies `normalize_total(1e4)` + `log1p` itself. Handing it an already-normalized matrix
would double-transform silently, and nothing downstream would raise. The transform applied here is
character-for-character the one the zero-shot baseline was computed under, because detector #1
compares against that baseline and any difference in the input transform would be indistinguishable
from CPT drift.


In [ ]:
import gc
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp

try:
    import psutil
    def _ram(tag):
        m = psutil.virtual_memory()
        print(f"[RAM] {tag:28s}: {m.used/1e9:5.1f} / {m.total/1e9:.1f} GB ({m.percent:.0f}%)")
except ImportError:
    def _ram(tag): pass

sc.settings.verbosity = 1

MICRO_PATH = os.path.join(DRIVE_ROOT, "micro_subset", "micro_subset.h5ad")
ASTRO_PATH = os.path.join(DRIVE_ROOT, "astro_subset", "astro_subset.h5ad")
for p in (MICRO_PATH, ASTRO_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(f"missing labelled subset {p} (colab_07 / colab_08 output)")

micro = sc.read_h5ad(MICRO_PATH)
astro = sc.read_h5ad(ASTRO_PATH)
print("microglia subset:", micro.shape)
print("astrocyte subset:", astro.shape)
assert list(micro.var_names) == list(astro.var_names), "gene panels differ between subsets"

micro.obs["lineage"] = "microglia"
astro.obs["lineage"] = "astrocyte"
KEEP_OBS = ["lineage", "substate", "apoe_carrier", "study_id", "donor_id", "total_counts"]
micro.obs = micro.obs[[c for c in KEEP_OBS if c in micro.obs.columns]].copy()
astro.obs = astro.obs[[c for c in KEEP_OBS if c in astro.obs.columns]].copy()
glia = ad.concat([micro, astro], join="inner", index_unique="-")
del micro, astro; gc.collect()
glia.obs["cell_index"] = np.arange(glia.n_obs)
glia.var["gene_name"] = glia.var_names            # explicit symbol column for scGPT's gene lookup
print("\ncombined glia:", glia.shape)

# raw-counts guard -- scGPT's transform must START from raw counts.
_idx = np.random.default_rng(0).choice(glia.n_obs, size=min(2000, glia.n_obs), replace=False)
Xs = glia.X[_idx]
data = Xs.data if sp.issparse(Xs) else np.asarray(Xs).ravel()
frac_int = float(np.mean(np.mod(data, 1) == 0)) if data.size else 1.0
assert frac_int >= 0.99, f".X is not raw counts (int frac {frac_int:.3f}) -- FM input must be raw"
print("raw-counts int-frac:", round(frac_int, 3))

# Identical to the zero-shot baseline's input transform (colab_10 3a). Any deviation here would
# register downstream as drift that CPT did not cause.
sc.pp.normalize_total(glia, target_sum=1e4)
sc.pp.log1p(glia)
print("applied normalize_total(1e4) + log1p")
_ram("combined glia (normalized)")



### 3b — Schema and substrate reference numbers

Fail loud before anything expensive. The counts asserted here are the frozen substrate every FM
notebook in the project loads; a mismatch means an upstream file changed and this run would be
measuring drift against a baseline computed on different cells. These checks deliberately run
against the **full** object, before any subsampling, so a smoke run verifies them too.


In [ ]:
REF_N_CELLS  = 142588
REF_N_GENES  = 26514
REF_LINEAGE  = {"astrocyte": 87783, "microglia": 54805}

for col in ("lineage", "substate", "apoe_carrier", "study_id", "donor_id", "cell_index"):
    assert col in glia.obs.columns, f"missing obs column: {col}"
    n_null = int(glia.obs[col].isna().sum())
    assert n_null == 0, f"obs column {col!r} has {n_null} null values"

assert glia.n_obs == REF_N_CELLS, f"substrate cell count {glia.n_obs} != reference {REF_N_CELLS}"
assert glia.n_vars == REF_N_GENES, f"gene panel {glia.n_vars} != reference {REF_N_GENES}"
lin_counts = glia.obs["lineage"].value_counts().to_dict()
assert lin_counts == REF_LINEAGE, f"lineage counts {lin_counts} != reference {REF_LINEAGE}"
print(f"substrate OK: {glia.n_obs} cells x {glia.n_vars} genes | lineage {lin_counts}")

print("\nlineage x substate:")
print(pd.crosstab(glia.obs["lineage"], glia.obs["substate"]))
print("\napoe_carrier:", glia.obs["apoe_carrier"].value_counts(dropna=False).to_dict())
print("study_id:", glia.obs["study_id"].value_counts().to_dict())
print("donors:", glia.obs["donor_id"].nunique())



### 3c — Split verification against the frozen artifact, then the smoke subsample

The donor split is **verified, never redrawn**. `outputs/donor_split.json` was frozen when the
first CPT run was made and every regime and every FM has used it since; a notebook that quietly
redraws it would produce numbers that are not comparable to anything already recorded. The check
here asserts the seed, the balance margin, the donor counts, the cell counts, and — the part that
a count-only check would miss — the exact donor identities on each side.

The smoke subsample is applied last, after all of the above has been checked on the full object
and before the first expensive stage.


In [ ]:
SPLIT_PATH = os.path.join(REPO_PATH, "outputs", "donor_split.json")
assert os.path.exists(SPLIT_PATH), f"missing frozen split {SPLIT_PATH}"
with open(SPLIT_PATH) as f:
    split_artifact = json.load(f)

REF_SEED       = 32
REF_MARGIN     = 10
REF_N_DONORS   = {"train": 101, "val": 22, "test": 22}
REF_N_CELLS_SP = {"train": 94963, "val": 23824, "test": 23801}

assert int(split_artifact["seed"]) == REF_SEED, f"split seed {split_artifact['seed']} != {REF_SEED}"
assert int(split_artifact["test_worst_case_margin"]) == REF_MARGIN, "split test margin != reference"
assert {k: int(v) for k, v in split_artifact["n_donors"].items()} == REF_N_DONORS, "split donor counts != reference"

split_map = split_artifact["donor_split"]
glia.obs["split"] = glia.obs["donor_id"].astype(str).map(split_map)
assert not glia.obs["split"].isna().any(), "some substrate donors are absent from the frozen split"
glia.obs["split"] = glia.obs["split"].astype("category")

cell_counts = glia.obs["split"].value_counts().to_dict()
assert {k: int(v) for k, v in cell_counts.items()} == REF_N_CELLS_SP, (
    f"cells per split {cell_counts} != reference {REF_N_CELLS_SP}")

# Donor identity, not just donor count -- a count-only check passes on a reshuffled split.
n_mismatch = 0
for part in ("train", "val", "test"):
    frozen = {d for d, s in split_map.items() if s == part}
    here   = set(glia.obs.loc[glia.obs["split"] == part, "donor_id"].astype(str))
    if frozen != here:
        n_mismatch += len(frozen ^ here)
        print(f"  MISMATCH {part}: {sorted(frozen ^ here)[:5]}")
assert n_mismatch == 0, f"{n_mismatch} donors differ between the frozen split and this substrate"
print(f"split verified: seed {REF_SEED}, margin {REF_MARGIN}, donors {REF_N_DONORS}, "
      f"cells {cell_counts}, donor identities match for all {len(split_map)} donors")

test_by_study = glia.obs.loc[glia.obs["split"] == "test", "study_id"].value_counts(normalize=True)
print("test-set study fractions:", test_by_study.round(3).to_dict())

# --- smoke subsample (last, and only here) ------------------------------------------------
if SMOKE:
    rng = np.random.default_rng(0)
    keep = []
    grp = glia.obs.groupby(["split", "lineage", "substate"], observed=True).indices
    starved = []
    for k, idx in grp.items():
        take = min(SMOKE_CAP, len(idx))
        if take < SMOKE_CAP:
            starved.append((k, len(idx)))
        keep.append(rng.choice(idx, size=take, replace=False))
    keep = np.sort(np.concatenate(keep))
    glia = glia[keep].copy()
    print(f"\nSMOKE subsample: {glia.n_obs} cells from {len(grp)} groups (cap {SMOKE_CAP}/group)")
    if starved:
        print(f"  {len(starved)} group(s) below cap:", starved[:5])
    print("  split:", glia.obs["split"].value_counts().to_dict())
    _ram("after smoke subsample")


## 4 — Vocabulary and input geometry



### 4a — Vocabulary intersection, APOE gate, in-vocab gene subset

scGPT reads gene **symbols** directly — there is no Ensembl mapping step, unlike the Geneformer
path. The steps below reproduce, in order, what the library's own embedding entry point does when
it prepares an object: append the special tokens if the checkpoint's vocabulary lacks them, mark
each panel gene with its vocabulary id (`-1` when absent), drop the out-of-vocabulary genes, then
build the gene-id array **from the surviving genes in their surviving order**. That ordering is
load-bearing — the id array indexes the same columns the count matrix is sliced by, and a mismatch
would silently feed the model the wrong gene for every value.

APOE remains a pre-registered hard fail: without it the APOE-axis eval cannot be run for this FM
at all. The in-vocabulary fraction is cross-checked against the value the zero-shot run recorded,
recomputed here rather than read back, so this is a real agreement check and not a restatement.


In [ ]:
from scgpt.tokenizer import GeneVocab

VOCAB_FILE  = os.path.join(MODEL_DIR, "vocab.json")
CONFIG_FILE = os.path.join(MODEL_DIR, "args.json")
with open(CONFIG_FILE) as f:
    model_configs = json.load(f)
print("checkpoint args.json:", json.dumps(model_configs, indent=1))

PAD_TOKEN = "<pad>"
SPECIAL_TOKENS = [PAD_TOKEN, "<cls>", "<eoc>"]
vocab = GeneVocab.from_file(VOCAB_FILE)
for s in SPECIAL_TOKENS:
    if s not in vocab:
        vocab.append_token(s)
print("\nscGPT vocabulary size:", len(vocab))

glia.var["id_in_vocab"] = [vocab[g] if g in vocab else -1 for g in glia.var["gene_name"]]
n_total = glia.n_vars
n_vocab = int((glia.var["id_in_vocab"] >= 0).sum())
print(f"gene panel: {n_total} | in scGPT vocab: {n_vocab} ({n_vocab/n_total:.1%})")

NICHE_CRITICAL_GENES = ["APOE", "TREM2", "MS4A6A", "CLU", "GFAP", "AQP4", "AIF1", "CSF1R"]
panel = set(glia.var["gene_name"])
niche_status = {g: {"in_panel": g in panel, "in_vocab": bool(g in vocab)} for g in NICHE_CRITICAL_GENES}
print("\nniche-critical gene survival:")
for g, s in niche_status.items():
    flag = "OK" if s["in_vocab"] else ("WARN" if s["in_panel"] else "ABSENT")
    print(f"  {g:8s} panel={str(s['in_panel']):5} vocab={str(s['in_vocab']):5}  [{flag}]")
assert niche_status["APOE"]["in_vocab"], (
    "APOE is not in the scGPT vocabulary -- eval #2 cannot be run for this FM. Pre-registered hard fail.")
niche_warnings = [g for g, s in niche_status.items() if not s["in_vocab"]]
if niche_warnings:
    print("WARN -- niche genes not in vocab (logged to audit):", niche_warnings)

# Cross-check the recomputed vocabulary coverage against what the zero-shot run recorded. Cell
# subsampling does not touch the gene panel, so this holds under SMOKE too.
AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
with open(AUDIT_REPORT_PATH) as f:
    _report = json.load(f)
_zs_audit = _report["scgpt_zeroshot"]["vocab_audit"]
frac_here = round(n_vocab / n_total, 4)
assert frac_here == _zs_audit["frac_in_vocab"], (
    f"in-vocab fraction {frac_here} != zero-shot record {_zs_audit['frac_in_vocab']} -- "
    "the gene panel or the vocabulary changed; the baseline is no longer comparable")
print(f"\nvocab coverage agrees with the zero-shot record: {frac_here}")

# Subset to in-vocab genes, THEN build the id array from the surviving genes, in order.
glia_v = glia[:, glia.var["id_in_vocab"] >= 0].copy()
vocab.set_default_index(vocab[PAD_TOKEN])
GENE_IDS = np.array(vocab(glia_v.var["gene_name"].tolist()), dtype=int)
assert len(GENE_IDS) == glia_v.n_vars, "gene-id array is not aligned to the in-vocab gene panel"
assert (GENE_IDS >= 0).all(), "a surviving gene mapped to a negative vocabulary id"
CLS_ID  = vocab["<cls>"]
PAD_ID  = vocab[PAD_TOKEN]
PAD_VALUE = model_configs["pad_value"]
print(f"in-vocab matrix: {glia_v.shape} | <cls> id {CLS_ID} | <pad> id {PAD_ID} | pad_value {PAD_VALUE}")
_ram("in-vocab matrix")

VOCAB_AUDIT = {
    "vocab_size": len(vocab), "n_genes_panel": n_total, "n_in_vocab": n_vocab,
    "frac_in_vocab": frac_here, "niche_status": niche_status,
    "niche_warnings": niche_warnings, "apoe_hard_fail_gate": "passed",
}



### 4b — Sequence-length geometry: how much of the substrate exceeds the context window

scGPT's whole-human checkpoint has a 1,200-token context. A cell with more detected in-vocabulary
genes than that does not get truncated by rank — it gets a **random subset** of its genes on every
pass. The fraction of cells above the ceiling is therefore the direct driver of how stochastic the
embedding is, and it sets the scale of the noise floor §6a is about to measure. Printing it here,
before any GPU work, means the floor measured later can be read against a known cause instead of
being an unexplained number.

The batch geometry check is the same arithmetic that has bitten this project once before: an
attention score tensor is `batch x heads x length^2` elements, and a 32-bit element count is a real
ceiling. At this context length there is a wide margin, but the check is cheap and it fails before
the run rather than during it.


In [ ]:
MAX_LENGTH = 1200        # whole-human context; the value the zero-shot baseline used

# nnz per row straight off the CSR index pointer -- no densification. Explicit stored zeros
# (should not exist after log1p, but not assumed) are dropped first so this count matches
# exactly what GliaCellDataset.__getitem__'s np.nonzero(row) sees per cell downstream.
Xv = glia_v.X
assert sp.issparse(Xv), "expected a sparse in-vocab matrix; densifying the full object is not intended here"
Xv = Xv.tocsr()
Xv.eliminate_zeros()
nnz_per_cell = np.diff(Xv.indptr)
seq_len = nnz_per_cell + 1                       # +1 for the prepended <cls> token
over = int((seq_len > MAX_LENGTH).sum())

print(f"detected in-vocab genes per cell (n={len(nnz_per_cell)}):")
for q in (0.05, 0.25, 0.50, 0.75, 0.95, 1.00):
    print(f"  q{q:<5.2f}: {np.quantile(nnz_per_cell, q):8.0f}")
print(f"\ncells above the {MAX_LENGTH}-token context: {over} ({over/len(seq_len):.1%})")
print("  -> these are RANDOMLY SUBSAMPLED to the context length on every embedding pass, so the")
print("     same cell embeds to a different vector each time. Detector #1's noise floor is not")
print("     zero for this FM; it is measured in 6a and used as the gate in 7a.")
if over == 0:
    print("  NOTE: no cell exceeds the context -- embedding would then be deterministic and the")
    print("        measured floor should come back at essentially 0.")

zero_cells = int((nnz_per_cell == 0).sum())
assert zero_cells == 0, f"{zero_cells} cells have no detected in-vocab gene -- they cannot be embedded"

# int32 ceiling on the attention score tensor (batch x heads x len^2).
INT32MAX = 2**31 - 1
N_HEADS = model_configs["nheads"]
EMB_BATCH = 64
_elems = EMB_BATCH * N_HEADS * MAX_LENGTH**2
assert _elems < INT32MAX, (
    f"attention tensor {EMB_BATCH}x{N_HEADS}x{MAX_LENGTH}^2 = {_elems:,} exceeds the int32 limit; "
    "halve EMB_BATCH until it fits")
print(f"\nbatch geometry OK: {EMB_BATCH} x {N_HEADS} heads x {MAX_LENGTH}^2 = {_elems:,} < {INT32MAX:,}")


## 5 — Continued pretraining with LoRA



### 5a — Build the model, verify the checkpoint actually loaded, attach the LoRA adapter

**Model construction.** The model is built with exactly the argument list the library's own
embedding entry point uses, because the zero-shot baseline was produced through that path and the
post-CPT embedding has to be produced through an identical one for the comparison to mean anything.
Two of those arguments are choices rather than defaults and are worth naming: the value encoder is
the *continuous* one (binned values are fed as floats, not as categorical bins), and the
multi-view decoder is constructed but never used by this notebook's objective.

**Checkpoint loading is a silent-failure surface.** The loader is non-strict by design: any
parameter whose name or shape does not match the model is dropped without a word. That is exactly
how a run could end up continuing to pretrain against a *randomly initialised* expression decoder
and never notice — the loss would fall, the drift would be real, and the whole thing would be
meaningless. The check below reproduces the loader's own key-renaming rules, then asserts that
every encoder, value-encoder and decoder parameter actually received a pretrained weight, and
prints anything dropped.

**LoRA targets.** The Geneformer counterpart adapts the attention projections plus the dense
projections. In this architecture the attention projections are not separate modules: query, key
and value live in a single fused parameter inside the attention module, with no forward method to
hook. The adapter library handles this case explicitly by wrapping the attention module itself —
which covers the fused input projection *and* the output projection together — and its own
documentation warns that targeting the output projection by name instead produces an adapter that
is silently ignored, because that submodule's forward is never called. So the encoder-equivalent
target set here is the attention module plus the two feed-forward projections.

Note what this does **not** include: the expression decoder that produces the training loss is left
frozen and un-adapted. Every trainable parameter in this run sits upstream of the `<cls>` position
that the cell embedding is read from.


In [ ]:
import re
import torch
from scgpt.model import TransformerModel
from peft import LoraConfig, get_peft_model

DEVICE = torch.device("cuda")

# Seed BEFORE any weight initialisation happens -- LoRA's A matrices are Kaiming-initialised at
# get_peft_model() time below. Seeding only in 5b (as originally written) left that init unseeded,
# so the audit's `training_seed` overstated how reproducible the run actually was.
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

# PyTorch's attention modules carry an inference fast path that reads `self_attn.in_proj_weight`
# as a raw tensor instead of calling the module's forward. Every forward pass in this notebook
# runs under autocast, and both `nn.MultiheadAttention.forward` and
# `nn.TransformerEncoderLayer.forward` already skip that path on their own in that case -- this is
# disabled explicitly anyway so nothing here depends on that being true implicitly.
torch.backends.mha.set_fastpath_enabled(False)
print("attention fast path enabled:", torch.backends.mha.get_fastpath_enabled())

def build_scgpt(model_configs, vocab):
    """Construct the model with the same arguments the library's embedding path uses."""
    return TransformerModel(
        ntoken=len(vocab),
        d_model=model_configs["embsize"],
        nhead=model_configs["nheads"],
        d_hid=model_configs["d_hid"],
        nlayers=model_configs["nlayers"],
        nlayers_cls=model_configs["n_layers_cls"],
        n_cls=1,
        vocab=vocab,
        dropout=model_configs["dropout"],
        pad_token=model_configs["pad_token"],
        pad_value=model_configs["pad_value"],
        do_mvc=True, do_dab=False, use_batch_labels=False,
        domain_spec_batchnorm=False, explicit_zero_prob=False,
        use_fast_transformer=False,        # no flash-attn installed -> PyTorch attention, as in the baseline
        fast_transformer_backend="flash", pre_norm=False,
    )

# Rename rules the loader applies when the model is NOT using the fast transformer: the released
# checkpoint stores fused attention weights under the flash-attention naming.
RENAME_RULES = {
    r"self_attn\._impl\.Wqkv\.": "self_attn.in_proj_",
    r"self_attn\.Wqkv\.": "self_attn.in_proj_",
    r"self_attn\._impl\.out_proj\.": "self_attn.out_proj.",
}

def load_checkpoint_verbose(model, ckpt_path):
    """Load the checkpoint and REPORT what was dropped -- the loader itself drops silently."""
    raw = torch.load(ckpt_path, map_location="cpu")
    renamed = {}
    for k, v in raw.items():
        kk = k
        for pat, rep in RENAME_RULES.items():
            kk = re.sub(pat, rep, kk)
        renamed[kk] = v
    model_dict = model.state_dict()
    kept    = {k: v for k, v in renamed.items() if k in model_dict and v.shape == model_dict[k].shape}
    dropped = sorted(set(renamed) - set(kept))
    model_dict.update(kept)
    model.load_state_dict(model_dict)
    return kept, dropped

REQUIRED_PREFIXES = ("encoder.", "value_encoder.", "transformer_encoder.", "decoder.")

def assert_required_loaded(model, kept, tag):
    """Every parameter on the path from input to loss must have come from the checkpoint. A
    randomly initialised expression decoder would make the whole CPT objective meaningless
    without crashing. Reused for both the CPT model here and the base-model reload in 6a -- that
    reload previously only checked len(kept) matched, which can't fail by construction."""
    missing = [n for n, _ in model.named_parameters()
               if n.startswith(REQUIRED_PREFIXES) and n not in kept]
    assert not missing, (
        f"[{tag}] {len(missing)} parameter(s) on the input->loss path were NOT loaded from the "
        f"checkpoint, e.g. {missing[:8]} -- CPT against randomly initialised weights would be meaningless")
    print(f"[{tag}] all {len(REQUIRED_PREFIXES)} required parameter groups loaded from the checkpoint")

scgpt_model = build_scgpt(model_configs, vocab)
kept, dropped = load_checkpoint_verbose(scgpt_model, os.path.join(MODEL_DIR, "best_model.pt"))
print(f"checkpoint parameters loaded: {len(kept)} | dropped: {len(dropped)}")
if dropped:
    print("  dropped keys (not present in this model, or shape-mismatched):")
    for k in dropped[:20]:
        print("   ", k)
    if len(dropped) > 20:
        print(f"    ... and {len(dropped)-20} more")

assert_required_loaded(scgpt_model, kept, "CPT model")

# ---------------------------------------------------------------- LoRA
LORA_R, LORA_ALPHA, LORA_DROPOUT = 8, 16, 0.05
# A plain name list here is dangerous: peft matches list-form target_modules by NAME SUFFIX, and
# scGPT's ContinuousValueEncoder ALSO has attributes named `linear1`/`linear2` -- a list
# ["self_attn", "linear1", "linear2"] would silently pull the value encoder into the adapter too.
# A regex target (peft matches it with re.fullmatch against the full dotted module name) scopes
# this to the transformer encoder layers only, which is what "encoder attention + feed-forward,
# decoder untouched" is actually meant to mean.
LORA_TARGETS_REGEX = r"transformer_encoder\.layers\.\d+\.(self_attn|linear1|linear2)$"

lora_cfg = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS_REGEX, bias="none",
)
peft_model = get_peft_model(scgpt_model, lora_cfg)

# The adapter injection is in place, so `scgpt_model` itself now carries the LoRA layers -- assert
# it rather than assume it, because a target regex that matched nothing fails silently.
_layer0 = scgpt_model.transformer_encoder.layers[0]
for name, mod in (("self_attn", _layer0.self_attn), ("linear1", _layer0.linear1), ("linear2", _layer0.linear2)):
    assert hasattr(mod, "lora_A"), f"LoRA did not attach to {name} -- target regex matched nothing"

# And assert it did NOT also attach to the value encoder -- the exact failure mode the regex
# (instead of a plain name list) exists to prevent.
_value_encoder_leak = [n for n, m in scgpt_model.named_modules()
                       if n.startswith("value_encoder") and hasattr(m, "lora_A")]
assert not _value_encoder_leak, f"LoRA leaked into value_encoder despite the scoped regex: {_value_encoder_leak}"

# Expected trainable count, derived: per encoder layer the adapter adds
#   fused input projection  r*(d) + r*(3d) = 4rd     (in 512 -> out 3*512, one fused parameter)
#   attention output proj   r*(d + d)      = 2rd
#   feed-forward linear1    r*(d + d_hid)
#   feed-forward linear2    r*(d_hid + d)
D, D_HID, N_LAYERS = model_configs["embsize"], model_configs["d_hid"], model_configs["nlayers"]
expected_trainable = N_LAYERS * (4*LORA_R*D + 2*LORA_R*D + 2*LORA_R*(D + D_HID))
actual_trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
peft_model.print_trainable_parameters()
print(f"\nexpected trainable (derived): {expected_trainable:,} | actual: {actual_trainable:,}")
assert actual_trainable == expected_trainable, (
    "trainable-parameter count does not match the derivation -- the adapter did not attach where "
    "this notebook's accounting assumes it did")

n_frozen = sum(p.numel() for p in peft_model.parameters() if not p.requires_grad)
print(f"frozen base parameters: {n_frozen:,} | adapted fraction: {actual_trainable/(n_frozen+actual_trainable):.4%}")
peft_model.to(DEVICE)



### 5b — Dataset, masked-value collator, training configuration

Each cell becomes a variable-length sequence of its detected genes, with a `<cls>` token prepended
and that token's value set to the pad value. The collator then does three things in order: bin the
expression values into 51 quantile bins within the cell, sample or pad the sequence to the context
length, and replace a fraction of the values with the mask sentinel. The model receives the masked
values and is scored on reconstructing the unmasked ones — masked mean-squared error over exactly
the positions that were masked, which is the objective the checkpoint was pretrained under.

Rows are densified one at a time rather than materialising the whole in-vocabulary matrix, which
keeps memory flat and behaves the same at smoke scale and at full scale.

Three guards worth naming. The mask sentinel and the pad value must differ, or the loss would be
computed over padding; that is asserted rather than assumed. Validation is run with a fixed seed
and no worker processes, so the mask draw is reproducible across evaluation points — the
validation curve then reflects the model changing rather than the mask changing. And the periodic
eval in 5c does not run on the full val split: at 23,824 held-out cells evaluated eight times over
the run, a full-val schedule would add roughly 3x the training loop's own forward-pass volume on
top of training itself. A fixed-size (2,000-cell), fixed-seed subsample is drawn here instead,
stratified by lineage so the eval set keeps the same astrocyte/microglia composition as the full
val split. This only affects the training-time monitoring curve — detector #1 downstream
(6a/6b/7) always embeds the full substrate, never this subsample.

The training budget is a deliberate match to the Geneformer aggregated run: same effective batch,
same number of optimizer steps, over the same training donors, which works out to the same
fraction of an epoch. Whether that budget is *adequate* is a separate and still-open question; it
is held fixed here so that the FM is the thing being varied.

In [ ]:
from torch.utils.data import DataLoader, SequentialSampler, RandomSampler
from scgpt.data_collator import DataCollator
from scgpt.loss import masked_mse_loss

# ---------------------------------------------------------------- CPT configuration
MASK_VALUE     = -1        # the collator's mask sentinel
MLM_PROB       = 0.15
LEARNING_RATE  = 5e-4
PER_DEV_BATCH  = 8
GRAD_ACCUM     = 4         # effective batch 32
MAX_STEPS      = 8 if SMOKE else 2000
WARMUP_RATIO   = 0.05
EVAL_STEPS     = 4 if SMOKE else 250
LOG_STEPS      = 2 if SMOKE else 50
NUM_WORKERS    = 2
VAL_EVAL_CAP   = 2000       # periodic-eval subsample size (5c); full val split stays untouched
VAL_EVAL_SEED  = 0          # independent of SEED -- fixes which val cells are watched, not the mask draw
# SEED is set once in 5a, before LoRA weight init; reseeding here (same value) fixes the
# RandomSampler draw and the mask draws independent of what happened in between.

assert MASK_VALUE != PAD_VALUE, (
    f"mask sentinel {MASK_VALUE} equals the pad value {PAD_VALUE}; masked positions would include "
    "padding and the loss would be computed over it")

torch.manual_seed(SEED)
np.random.seed(SEED)

class GliaCellDataset(torch.utils.data.Dataset):
    """One cell -> (gene ids, expression values) with <cls> prepended, densified per row."""
    def __init__(self, X_csr, gene_ids, cls_id, pad_value):
        self.X, self.gene_ids, self.cls_id, self.pad_value = X_csr, gene_ids, cls_id, pad_value
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        row = self.X[idx].toarray().ravel()
        nz = np.nonzero(row)[0]
        genes  = np.insert(self.gene_ids[nz], 0, self.cls_id)
        values = np.insert(row[nz], 0, self.pad_value)
        return {"id": idx,
                "genes": torch.from_numpy(genes).long(),
                "expressions": torch.from_numpy(values).float()}

full_ds = GliaCellDataset(Xv, GENE_IDS, CLS_ID, PAD_VALUE)
split_vals = glia_v.obs["split"].astype(str).values
train_idx = np.flatnonzero(split_vals == "train")
val_idx   = np.flatnonzero(split_vals == "val")
assert len(train_idx) > 0 and len(val_idx) > 0, "empty train or val split"
train_ds = torch.utils.data.Subset(full_ds, train_idx)

# Periodic eval (5c) does not run on the full val split -- at len(val_idx) cells evaluated
# MAX_STEPS/EVAL_STEPS times, that would add several times the training loop's own forward-pass
# volume on top of training itself. A fixed-size, fixed-seed subsample is drawn instead,
# stratified by lineage so it keeps the same composition as the full val split. This only
# affects the training-time monitoring curve -- detector #1 (6a/6b/7) always embeds every
# cell in the full substrate, never this subsample.
lineage_val = glia_v.obs["lineage"].astype(str).values[val_idx]
rng_val_eval = np.random.default_rng(VAL_EVAL_SEED)
val_eval_parts = []
for lin in sorted(np.unique(lineage_val)):
    lin_idx = val_idx[lineage_val == lin]
    take = min(int(round(VAL_EVAL_CAP * len(lin_idx) / len(val_idx))), len(lin_idx))
    val_eval_parts.append(rng_val_eval.choice(lin_idx, size=take, replace=False))
val_eval_idx = np.sort(np.concatenate(val_eval_parts))
assert len(val_eval_idx) > 0, "val eval subsample is empty"

val_ds   = torch.utils.data.Subset(full_ds, val_eval_idx)
print(f"cells -> train {len(train_ds)} | val eval subsample {len(val_ds)} of {len(val_idx)} "
      f"full val split (cap {VAL_EVAL_CAP}, seed {VAL_EVAL_SEED}) | total {len(full_ds)}")

def make_collator(do_mlm):
    return DataCollator(
        do_padding=True, pad_token_id=PAD_ID, pad_value=PAD_VALUE,
        do_mlm=do_mlm, do_binning=True, mlm_probability=MLM_PROB,
        mask_value=MASK_VALUE, max_length=MAX_LENGTH,
        sampling=True, keep_first_n_tokens=1,
    )

train_loader = DataLoader(train_ds, batch_size=PER_DEV_BATCH,
                          sampler=RandomSampler(train_ds), collate_fn=make_collator(True),
                          drop_last=True, num_workers=NUM_WORKERS, pin_memory=True)
# num_workers=0 + a reseed before each pass -> the validation mask draw is identical every time.
val_loader   = DataLoader(val_ds, batch_size=PER_DEV_BATCH,
                          sampler=SequentialSampler(val_ds), collate_fn=make_collator(True),
                          drop_last=False, num_workers=0, pin_memory=True)

n_epochs_equiv = MAX_STEPS * PER_DEV_BATCH * GRAD_ACCUM / max(len(train_ds), 1)
print(f"budget: {MAX_STEPS} steps x eff batch {PER_DEV_BATCH*GRAD_ACCUM} "
      f"= {n_epochs_equiv:.3f} epochs over the training split | lr {LEARNING_RATE}")


### 5c — Run CPT and save the LoRA adapter

A plain training loop: gradient accumulation to the effective batch, linear warmup then linear
decay, bfloat16 for the forward pass with the loss taken in float32. The validation pass runs on a
fixed schedule, over the fixed-size subsample drawn in 5b, and its curve is captured in full, so
the question of whether the run was still improving when the budget ran out can be answered from
the record rather than re-derived later.

In [ ]:
import math, time
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

optimizer = AdamW([p for p in peft_model.parameters() if p.requires_grad], lr=LEARNING_RATE)
warmup_steps = max(1, int(WARMUP_RATIO * MAX_STEPS))

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    return max(0.0, (MAX_STEPS - step) / max(1, MAX_STEPS - warmup_steps))

scheduler = LambdaLR(optimizer, lr_lambda)
USE_BF16 = torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print("autocast dtype:", AMP_DTYPE)

def forward_loss(batch):
    gene = batch["gene"].to(DEVICE, non_blocking=True)
    target = batch["expr"].to(DEVICE, non_blocking=True)
    masked = batch["masked_expr"].to(DEVICE, non_blocking=True)
    pad_mask = gene.eq(PAD_ID)
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
        out = scgpt_model(src=gene, values=masked, src_key_padding_mask=pad_mask,
                          CLS=False, CCE=False, MVC=False, ECS=False)
    positions = masked.eq(MASK_VALUE)
    if positions.sum() == 0:
        return None, 0
    return masked_mse_loss(out["mlm_output"].float(), target.float(), positions), int(positions.sum())

@torch.no_grad()
def evaluate():
    scgpt_model.eval()
    torch.manual_seed(1234)          # fixed mask draw -> the val curve is comparable across steps
    tot, n = 0.0, 0
    for batch in val_loader:
        loss, npos = forward_loss(batch)
        if loss is None:
            continue
        tot += float(loss) * npos
        n += npos
    scgpt_model.train()
    return tot / max(n, 1)

log_history = []
scgpt_model.train()
step, running, running_n, t0 = 0, 0.0, 0, time.time()
train_iter = iter(train_loader)
print(f"training for {MAX_STEPS} steps (warmup {warmup_steps})")

while step < MAX_STEPS:
    optimizer.zero_grad(set_to_none=True)
    acc_loss = 0.0
    for _ in range(GRAD_ACCUM):
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)
        loss, npos = forward_loss(batch)
        if loss is None:
            continue
        (loss / GRAD_ACCUM).backward()
        acc_loss += float(loss) / GRAD_ACCUM
    torch.nn.utils.clip_grad_norm_([p for p in peft_model.parameters() if p.requires_grad], 1.0)
    optimizer.step()
    scheduler.step()
    step += 1
    running += acc_loss; running_n += 1

    if step % LOG_STEPS == 0:
        rec = {"step": step, "train_loss": running / running_n, "lr": scheduler.get_last_lr()[0]}
        log_history.append(rec)
        print(f"  step {step:5d} | train {rec['train_loss']:.4f} | lr {rec['lr']:.2e} "
              f"| {time.time()-t0:.0f}s")
        running, running_n = 0.0, 0
    if step % EVAL_STEPS == 0 or step == MAX_STEPS:
        vl = evaluate()
        log_history.append({"step": step, "eval_loss": vl})
        print(f"  step {step:5d} | VAL masked-MSE {vl:.4f}")

TRAIN_SECONDS = time.time() - t0
train_losses = [r["train_loss"] for r in log_history if "train_loss" in r]
eval_losses  = [(r["step"], r["eval_loss"]) for r in log_history if "eval_loss" in r]
TRAIN_LOSS_MEAN  = float(np.mean(train_losses)) if train_losses else float("nan")
TRAIN_LOSS_FINAL = float(train_losses[-1]) if train_losses else float("nan")
EVAL_LOSS_FINAL  = float(eval_losses[-1][1]) if eval_losses else float("nan")
print(f"\nfinished in {TRAIN_SECONDS/60:.1f} min | train_loss mean {TRAIN_LOSS_MEAN:.4f} "
      f"final {TRAIN_LOSS_FINAL:.4f} | final val {EVAL_LOSS_FINAL:.4f}")
print("val curve:", [(s, round(v, 4)) for s, v in eval_losses])

ADAPTER_DIR = os.path.join(DRIVE_ROOT, "scgpt", f"cpt_aggregated_{RUN_TAG}_adapter{SUFFIX}")
os.makedirs(ADAPTER_DIR, exist_ok=True)
peft_model.save_pretrained(ADAPTER_DIR)
print("saved LoRA adapter ->", ADAPTER_DIR)


## 6 — Re-embed the substrate



### 6a — Measure the noise floor: re-embed the frozen base model through this notebook's own path

Detector #1 asks whether the post-CPT embedding differs from the zero-shot one. For that question
to have an answer, the amount two embeddings differ *when nothing has changed* has to be known —
and for this FM that quantity is not zero, because cells above the context length are randomly
subsampled on every pass.

Two comparisons are made, and the pair is what makes them interpretable:

- **floor against the stored baseline** — a freshly built, unmodified base model embedded through
  this notebook's code, compared to the zero-shot file the baseline notebook wrote. This is the
  exact comparison detector #1 will make, minus the CPT.
- **floor against a repeat pass** — the same unmodified model embedded twice here, compared to
  itself. This isolates the sampling stochasticity alone.

If the two agree, the embedding path reimplemented in this notebook is faithful to the one that
produced the baseline, and the floor is sampling noise. If the first is materially larger than the
second, the difference is a fidelity problem in this notebook, not drift — and detector #1 must not
be read until it is fixed.


In [ ]:
@torch.no_grad()
def embed_cells(model, dataset, batch_size=EMB_BATCH, num_workers=NUM_WORKERS, desc=""):
    """<cls>-position cell embeddings, L2-normalized -- the same readout as the baseline."""
    from tqdm.auto import tqdm
    collator = make_collator(do_mlm=False)
    loader = DataLoader(dataset, batch_size=batch_size, sampler=SequentialSampler(dataset),
                        collate_fn=collator, drop_last=False,
                        num_workers=num_workers, pin_memory=True)
    model.eval()
    out = np.zeros((len(dataset), model_configs["embsize"]), dtype=np.float32)
    count = 0
    for batch in tqdm(loader, desc=desc or "embedding"):
        gene = batch["gene"].to(DEVICE, non_blocking=True)
        expr = batch["expr"].to(DEVICE, non_blocking=True)
        pad_mask = gene.eq(PAD_ID)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            h = model._encode(gene, expr, src_key_padding_mask=pad_mask, batch_labels=None)
        e = h[:, 0, :].float().cpu().numpy()          # <cls> position
        out[count:count + len(e)] = e
        count += len(e)
    assert count == len(dataset), f"embedded {count} of {len(dataset)} cells"
    return out / (np.linalg.norm(out, axis=1, keepdims=True) + 1e-12)

def per_cell_cosine(A, B):
    return (A * B).sum(1) / (np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1) + 1e-12)

# --- the stored zero-shot baseline, aligned by cell_index -------------------------------------
ZEROSHOT_PATH = os.path.join(DRIVE_ROOT, "scgpt", "glia_scgpt_zeroshot.h5ad")
assert os.path.exists(ZEROSHOT_PATH), f"missing zero-shot baseline {ZEROSHOT_PATH}"
zs = ad.read_h5ad(ZEROSHOT_PATH)
zs_X = zs.X.toarray() if sp.issparse(zs.X) else np.asarray(zs.X)
zs_df = pd.DataFrame(np.asarray(zs_X, dtype=np.float32), index=zs.obs["cell_index"].values)
zs_aligned = zs_df.reindex(glia_v.obs["cell_index"].values)
assert zs_aligned.notna().all().all(), "baseline rows missing after cell_index alignment"
X_ZS_STORED = zs_aligned.to_numpy(dtype=np.float32)

# cell_index alone can only catch a MISSING cell (both files use arange(n_obs), so a reindex
# against a permuted-but-complete index set would still pass the notna check above with no
# warning). Cross-check the labels that actually travel on both files at every aligned row --
# this is what would catch, e.g., a concatenation-order change upstream (astro before micro).
LABEL_COLS = ["lineage", "substate", "apoe_carrier", "study_id", "donor_id"]
zs_obs_aligned = zs.obs.set_index("cell_index").reindex(glia_v.obs["cell_index"].values)
for col in LABEL_COLS:
    if col not in zs_obs_aligned.columns:
        continue
    mismatch = (zs_obs_aligned[col].astype(str).values != glia_v.obs[col].astype(str).values)
    n_mismatch = int(mismatch.sum())
    assert n_mismatch == 0, (
        f"{n_mismatch} cells have a different '{col}' between the zero-shot baseline and this "
        "substrate after cell_index alignment -- the two files disagree about which row is which cell")
print("zero-shot baseline labels agree with this substrate at every aligned row "
      f"({', '.join(LABEL_COLS)})")

del zs, zs_X, zs_df, zs_aligned, zs_obs_aligned; gc.collect()
print("stored zero-shot baseline aligned:", X_ZS_STORED.shape)

# --- a fresh, unmodified base model ----------------------------------------------------------
base_model = build_scgpt(model_configs, vocab)
_k, _d = load_checkpoint_verbose(base_model, os.path.join(MODEL_DIR, "best_model.pt"))
assert_required_loaded(base_model, _k, "base reload")
base_model.to(DEVICE)

test_mask = (glia_v.obs["split"] == "test").values
assert test_mask.any(), "no test-split cells present -- cannot measure the noise floor on held-out cells"

X_BASE_A = embed_cells(base_model, full_ds, desc="base pass A")
X_BASE_B = embed_cells(base_model, full_ds, desc="base pass B")
del base_model; gc.collect(); torch.cuda.empty_cache()

floor_stored_all  = 1.0 - float(np.median(per_cell_cosine(X_ZS_STORED, X_BASE_A)))
floor_stored_test = 1.0 - float(np.median(per_cell_cosine(X_ZS_STORED, X_BASE_A)[test_mask]))
floor_repeat_all  = 1.0 - float(np.median(per_cell_cosine(X_BASE_A, X_BASE_B)))
floor_repeat_test = 1.0 - float(np.median(per_cell_cosine(X_BASE_A, X_BASE_B)[test_mask]))

print(f"\nnoise floor vs STORED baseline : all {floor_stored_all:.5f} | test {floor_stored_test:.5f}")
print(f"noise floor vs REPEAT pass     : all {floor_repeat_all:.5f} | test {floor_repeat_test:.5f}")
excess = floor_stored_test - floor_repeat_test
print(f"excess (stored - repeat, test) : {excess:+.5f}")
FIDELITY_TOL = 0.002
FIDELITY_OK = abs(excess) <= FIDELITY_TOL
if not FIDELITY_OK:
    print("  WARNING -- the stored-baseline floor exceeds the repeat floor by more than the")
    print(f"  tolerance ({FIDELITY_TOL}). That gap is a difference between this notebook's embedding")
    print("  path and the one that produced the baseline, NOT drift. Detector #1 in 7a is not")
    print("  interpretable until it is explained -- this is recorded (fidelity_ok=False) in the")
    print("  audit trail rather than left as a print-only warning, so a downstream reader of")
    print("  audit_report.json cannot miss it.")
else:
    print("  the two floors agree -> this notebook's embedding path reproduces the baseline;")
    print("  the floor is sampling stochasticity.")

NOISE_FLOOR = max(floor_stored_test, floor_repeat_test)   # the conservative gate for detector #1
print(f"\ndetector #1 noise floor (held-out test, conservative): {NOISE_FLOOR:.5f}")
_ram("after base embedding passes")



### 6b — Merge the adapter and embed the adapted model

The adapter is folded into the base weights before embedding rather than embedded through the
adapter wrapper. This is not a convenience: the attention adapter works by merging its weights
around each forward call, and PyTorch's attention module has an inference fast path that reads the
fused weight tensor directly instead of calling the module's forward. Embedding through the wrapper
would therefore risk producing a vector with the adaptation silently absent. Merging first removes
the question — after the merge there is only one plain set of weights, and it is the adapted one.


In [ ]:
merged = peft_model.merge_and_unload()
merged.to(DEVICE)

# After the merge there must be no adapter machinery left anywhere in the graph.
leftover = [n for n, m in merged.named_modules() if hasattr(m, "lora_A")]
assert not leftover, f"LoRA layers survived merge_and_unload: {leftover[:5]}"
_l0 = merged.transformer_encoder.layers[0]
assert isinstance(_l0.self_attn, torch.nn.MultiheadAttention), "attention module is not a plain MultiheadAttention after merge"
print("adapter merged into the base weights")

X_CPT = embed_cells(merged, full_ds, desc="CPT embedding")
assert X_CPT.shape == X_BASE_A.shape, "post-CPT embedding shape differs from the base embedding"
assert np.isfinite(X_CPT).all(), "post-CPT embedding contains non-finite values"
print("X_scGPT_cpt:", X_CPT.shape)

del merged; gc.collect(); torch.cuda.empty_cache()
_ram("after CPT embedding")


## 7 — Detector #1: is the embedding changing?



### 7a — Median per-cell cosine drift, gated on the measured floor

Drift is the median per-cell cosine distance from the stored zero-shot embedding to the post-CPT
embedding, reported on the held-out test donors (the contract's primary surface) with the
full-substrate figure alongside for context.

The verdict is *inert* or *real*, and nothing stronger. Detector #1 does not license calling
anything a win — it licenses proceeding to the evals. What is different from the Geneformer run is
the floor it is judged against: measured in this run, from this run's own stochasticity, rather
than inherited.


In [ ]:
assert test_mask.any(), "no test-split cells present -- cannot gate detector #1 on held-out"
cos_cpt = per_cell_cosine(X_ZS_STORED, X_CPT)
drift_all  = 1.0 - float(np.median(cos_cpt))
drift_test = 1.0 - float(np.median(cos_cpt[test_mask]))

inert = drift_test <= NOISE_FLOOR
ratio = drift_test / NOISE_FLOOR if NOISE_FLOOR > 0 else float("inf")

print(f"drift (all cells, context)  : {drift_all:.5f}")
print(f"drift (held-out test, GATE) : {drift_test:.5f}")
print(f"measured noise floor (test) : {NOISE_FLOOR:.5f}")
print(f"drift / floor               : {ratio:.2f}x")
print(f"\ndetector #1: {'INERT -- at or below the measured floor; the run is not interpretable' if inert else 'REAL (above the measured floor)'}")
if not inert:
    print("  'real' licenses proceeding to the evals only -- it is not itself a win.")
if not FIDELITY_OK:
    print("  CAUTION -- 6a's fidelity check did NOT pass (this notebook's embedding path disagrees")
    print("  with the stored zero-shot baseline by more than tolerance); this verdict may reflect")
    print("  that gap rather than real drift. See fidelity_ok in the audit entry.")

# Per-lineage, for a first look at whether the movement is uniform across the substrate -- kept
# (not just printed) so 7b can score each lineage's drift against ITS OWN substate reference,
# instead of the pooled, both-lineages drift_test figure.
DRIFT_TEST_BY_LINEAGE = {}
print("\ndrift by lineage (held-out test):")
for lin in sorted(glia_v.obs["lineage"].astype(str).unique()):
    m = test_mask & (glia_v.obs["lineage"].astype(str) == lin).values
    if m.sum() == 0:
        continue
    d = 1.0 - float(np.median(cos_cpt[m]))
    DRIFT_TEST_BY_LINEAGE[lin] = d
    print(f"  {lin:10s} n={int(m.sum()):6d} | drift {d:.5f}")

DETECTOR1 = {
    "drift_all": drift_all, "drift_test": drift_test,
    "drift_test_by_lineage": DRIFT_TEST_BY_LINEAGE,
    "noise_floor_measured": NOISE_FLOOR,
    "noise_floor_vs_stored_test": floor_stored_test,
    "noise_floor_vs_repeat_test": floor_repeat_test,
    "floor_excess_stored_minus_repeat": excess,
    "fidelity_ok": bool(FIDELITY_OK),
    "drift_over_floor": ratio,
    "gate": "held-out test", "inert": bool(inert),
}



### 7b — The magnitude anchor, recomputed in scGPT's own space

A drift number in cosine-distance units means nothing on its own; it needs a reference for what a
*biologically meaningful* change looks like in the same space. The project's existing anchor — the
within-donor distance between a lineage's two substate poles — was measured in the other FM's
embedding space and cannot be carried across.

It is recomputed here from the stored zero-shot embedding, the same way it was originally defined:
within each donor, between that donor's own cells at each pole, so that donor and study identity
cannot inflate it; median across donors that carry enough cells at both poles. The pooled figure
is printed next to it, because the gap between the two is itself the measure of how much the
confound was inflating the anchor.


In [ ]:
POLES = {"microglia": ("homeostatic", "activated"), "astrocyte": ("resting", "reactive")}
MIN_CELLS_PER_POLE = 50
MIN_DONORS = 5
PAIR_SAMPLE = 3000

lineage_v = glia_v.obs["lineage"].astype(str).values
substate_v = glia_v.obs["substate"].astype(str).values
donor_v = glia_v.obs["donor_id"].astype(str).values

def median_pairwise_cos_dist(Xa, Xb, n=PAIR_SAMPLE, seed=0):
    rng = np.random.default_rng(seed)
    ia = rng.choice(len(Xa), size=min(n, len(Xa)), replace=False)
    ib = rng.choice(len(Xb), size=min(n, len(Xb)), replace=False)
    Ua = Xa[ia] / (np.linalg.norm(Xa[ia], axis=1, keepdims=True) + 1e-12)
    Ub = Xb[ib] / (np.linalg.norm(Xb[ib], axis=1, keepdims=True) + 1e-12)
    return float(1.0 - np.median(Ua @ Ub.T))

SUBSTATE_REF = {}
for lin, (p1, p2) in POLES.items():
    m1_all = (lineage_v == lin) & (substate_v == p1)
    m2_all = (lineage_v == lin) & (substate_v == p2)
    if m1_all.sum() == 0 or m2_all.sum() == 0:
        print(f"{lin}: a substate pole is absent in this object -- reference not computed")
        SUBSTATE_REF[lin] = None
        continue
    pooled = median_pairwise_cos_dist(X_ZS_STORED[m1_all], X_ZS_STORED[m2_all])
    within = []
    for d in np.unique(donor_v):
        m1 = m1_all & (donor_v == d)
        m2 = m2_all & (donor_v == d)
        if m1.sum() >= MIN_CELLS_PER_POLE and m2.sum() >= MIN_CELLS_PER_POLE:
            within.append(median_pairwise_cos_dist(X_ZS_STORED[m1], X_ZS_STORED[m2]))
    print(f"\n{lin}: {p1} vs {p2}")
    print(f"  pooled pairwise cos-dist          : {pooled:.4f}  (n={int(m1_all.sum())}/{int(m2_all.sum())})")
    print(f"  qualifying donors (>={MIN_CELLS_PER_POLE} each pole): {len(within)}")
    if len(within) < MIN_DONORS:
        print(f"  UNSTABLE -- fewer than {MIN_DONORS} donors carry both poles; the within-donor median")
        print("  is not trustworthy here and the pooled figure (confound-inflated) is all there is.")
        SUBSTATE_REF[lin] = {"within_donor": None, "pooled": pooled, "n_donors": len(within)}
        continue
    wd = float(np.median(within))
    q25, q75 = float(np.quantile(within, 0.25)), float(np.quantile(within, 0.75))
    print(f"  within-donor pairwise cos-dist    : median {wd:.4f} (IQR {q25:.4f}-{q75:.4f})")
    print(f"  pooled minus within-donor         : {pooled - wd:+.4f}  (how much pooling inflates it)")
    SUBSTATE_REF[lin] = {"within_donor": wd, "pooled": pooled, "n_donors": len(within),
                         "iqr": [q25, q75]}

print("\ndetector #1 drift as a fraction of a meaningful change, in scGPT's own space:")
PCT_OF_REF = {}
for lin, ref in SUBSTATE_REF.items():
    if not ref:
        continue
    anchor = ref["within_donor"] if ref["within_donor"] is not None else ref["pooled"]
    kind = "within-donor" if ref["within_donor"] is not None else "POOLED (fallback)"
    lin_drift = DRIFT_TEST_BY_LINEAGE.get(lin, drift_test)   # per-lineage drift, not the pooled figure
    PCT_OF_REF[lin] = round(100 * lin_drift / anchor, 1) if anchor > 0 else None
    print(f"  {lin:10s} drift {lin_drift:.5f} = {PCT_OF_REF[lin]}% of the {kind} substate reference "
          f"({anchor:.4f})")
print("\nReported without a pass bar, by contract: detector #1 is diagnostic, and the only")
print("defensible anchors are the measured floor above and this reference.")


## 8 — Save + handoff



### 8a — Save the post-CPT embedding, append the audit trace, print the commit commands

The saved embedding carries every label the evals slice on, keyed by `cell_index`, so the eval
notebook needs nothing from this one but the file and the audit entry. A smoke run writes its
artifacts under the suffixed paths but does **not** touch the audit report — plumbing rehearsals do
not enter the record.


In [ ]:
import shlex

SCGPT_OUT_DIR = os.path.join(DRIVE_ROOT, "scgpt")
os.makedirs(SCGPT_OUT_DIR, exist_ok=True)

emb_adata = ad.AnnData(
    X=X_CPT,
    obs=glia_v.obs[["cell_index", "split", "lineage", "substate",
                    "apoe_carrier", "study_id", "donor_id"]].copy(),
)
EMB_PATH = os.path.join(SCGPT_OUT_DIR, f"glia_scgpt_cpt_aggregated_{RUN_TAG}{SUFFIX}.h5ad")
emb_adata.write_h5ad(EMB_PATH)
print("saved post-CPT embedding ->", EMB_PATH, f"({os.path.getsize(EMB_PATH)/1e9:.2f} GB)")

AUDIT_ENTRY = {
    "status": "computed", "date": TODAY, "regime": "aggregated", "fm": "scgpt",
    "training_seed": SEED, "donor_split_seed": REF_SEED,
    "model_dir": os.path.basename(MODEL_DIR), "scgpt_commit": SCGPT_COMMIT,
    "peft_version": peft.__version__,
    "n_cells": int(glia_v.n_obs),
    "n_train_cells": int((glia_v.obs["split"] == "train").sum()),
    "emb_dim": int(X_CPT.shape[1]),
    "max_length": MAX_LENGTH,
    "frac_cells_over_context": round(float(over) / len(seq_len), 4),
    "vocab_audit": VOCAB_AUDIT,
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT,
             "targets": LORA_TARGETS_REGEX, "trainable_params": int(actual_trainable),
             "frozen_params": int(n_frozen)},
    "train": {"max_steps": MAX_STEPS, "grad_accum": GRAD_ACCUM, "batch": PER_DEV_BATCH,
              "effective_batch": PER_DEV_BATCH * GRAD_ACCUM, "lr": LEARNING_RATE,
              "warmup_ratio": WARMUP_RATIO, "mlm_prob": MLM_PROB,
              "eval_steps": EVAL_STEPS, "val_eval_cap": VAL_EVAL_CAP,
              "val_eval_seed": VAL_EVAL_SEED, "n_val_eval_cells": int(len(val_eval_idx)),
              "n_val_full_cells": int(len(val_idx)),
              "epochs_equiv": round(float(n_epochs_equiv), 4),
              "train_loss_mean": round(TRAIN_LOSS_MEAN, 4),
              "train_loss_final": round(TRAIN_LOSS_FINAL, 4),
              "eval_loss_final": round(EVAL_LOSS_FINAL, 4),
              "log_history": log_history,
              "wall_clock_seconds": round(TRAIN_SECONDS, 1)},
    "detector_1": DETECTOR1,
    "substate_reference_scgpt_space": SUBSTATE_REF,
    "drift_pct_of_substate_reference": PCT_OF_REF,
    "donor_split_file": os.path.relpath(SPLIT_PATH, REPO_PATH),
    "adapter_file": os.path.relpath(ADAPTER_DIR, DRIVE_ROOT),
    "embedding_file": os.path.relpath(EMB_PATH, DRIVE_ROOT),
    "zeroshot_baseline_file": os.path.relpath(ZEROSHOT_PATH, DRIVE_ROOT),
    "note": ("detector #1 is gated on a floor measured in THIS run, not the Geneformer 0.0000 -- "
             "scGPT randomly subsamples cells above its context length, so repeat embedding is "
             "not deterministic. The substate magnitude anchor is likewise recomputed in scGPT's "
             "own embedding space and is not comparable to the Geneformer values."),
}

if SMOKE:
    print("\nSMOKE run -- audit_report.json NOT written. Entry that would have been added:")
    print(json.dumps({k: v for k, v in AUDIT_ENTRY.items() if k != "train"}, indent=2)[:2500])
else:
    with open(AUDIT_REPORT_PATH) as f:
        report = json.load(f)
    report["scgpt_cpt_aggregated"] = AUDIT_ENTRY
    with open(AUDIT_REPORT_PATH, "w") as f:
        json.dump(report, f, indent=2)
    print("audit trace appended ->", AUDIT_REPORT_PATH)

    rel = [os.path.relpath(p, REPO_PATH) for p in (FREEZE_PATH, ENV_JSON_PATH, AUDIT_REPORT_PATH)]
    print("\n=== Commit + push (from WSL -- Colab has no git creds) ===")
    print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git add " + " ".join(shlex.quote(r) for r in rel))
    print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git commit -m "
          "'colab_16: scGPT CPT (aggregated regime) + detector #1 on a measured noise floor'")
    print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git push")



### Carried forward

- The LoRA adapter and the post-CPT embedding, on Drive, keyed by `cell_index`.
- `outputs/audit_report.json["scgpt_cpt_aggregated"]` — detector #1 with its **measured** floor,
  the scGPT-space substate reference, the full validation curve, and the LoRA accounting.
- The next notebook runs evals #1 and #2 on this checkpoint against the scGPT zero-shot baseline,
  mirroring what was done for the other FM's aggregated run.
